# ENERGY / 2030: clean analysis
This notebook reproduces the public portfolio pipeline without local file paths or student identifiers.


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score


In [ ]:
DATA_URL = 'https://owid-public.owid.io/data/energy/owid-energy-data.csv'
df = pd.read_csv(DATA_URL)
df.head()


## Reuse the application preprocessing


In [ ]:
from src.config import ASIAN_COUNTRIES, MODEL_FEATURES, TARGET, FOSSIL_DOMINANT_THRESHOLD
cols = ['country', 'year', *MODEL_FEATURES, TARGET, 'renewables_share_energy']
asia = df.loc[df['country'].isin(ASIAN_COUNTRIES) & df['year'].between(2012, 2024), cols].copy()
asia['fossil_dominant'] = asia['fossil_share_energy'] > FOSSIL_DOMINANT_THRESHOLD
model_df = asia.dropna(subset=[*MODEL_FEATURES, TARGET]).copy()
model_df.shape


## Random Forest regression


In [ ]:
X = model_df[MODEL_FEATURES]
y = model_df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
reg = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)
print('Test R2:', round(r2_score(y_test, pred), 4))


## Fossil-dominance classification


In [ ]:
yc = model_df['fossil_dominant'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, yc, test_size=0.2, random_state=42, stratify=yc)
clf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
print('Classification accuracy:', round(accuracy_score(y_test, clf.predict(X_test)), 4))
